# FedTT / FedTT+ 再現実験：全ステップ（Colab GPU 版）

対象論文：Ghiasvand et al., Findings of ACL 2025。**「ランタイム → すべてのセルを実行」を押すだけ**で、安いものから順に進みます。コードはこのノートブックの中にあり、アップロードは不要です。途中で止まっても、もう一度「すべてのセルを実行」で続きから進みます。表 1 の MRPC だけなら `FedTT_MRPC_colab.ipynb` のほうが軽いです（同じ結果フォルダを使うので、どちらで回したランも共有されます）。

| 順 | ステップ | 論文 | ラン数 | 目安（A100、見込み） |
|---|---|---|---|---|
| 0 | パラメータ数・通信量の照合（学習なし） | 表 1, 2, 7, 12, 14, 15 | – | 1〜2 分 |
| 1 | スモークテスト | – | 5 | 5〜10 分 |
| 2 | 表 1：MRPC（学習率 4 × バッチ 2 の全グリッド、100 ラウンド） | 表 1 | 16 | 1〜2 時間 |
| 3 | 表 7：SST-2・QNLI（ランク 2/5/10 と FedTT+、20 ラウンド） | 表 7（と表 1） | 8 | 3〜5 時間 |
| 4 | 表 3：3 クライアント（iid） | 表 3 | 8 | 1〜2 時間 |
| 5 | 表 7：QQP・MNLI | 表 7（と表 1） | 8 | 10〜17 時間 |
| 6 | 表 3：20・50 クライアント（mild・severe） | 表 3, 図 2, 表 15 | 16 | 25〜35 時間 |
| 7 | 表 2：RoBERTa-base（FedPETuning の設定、学習率は 2 点に絞る） | 表 2 | 10 | 5〜10 時間 |
| 8 | 表 13：3 クライアント・ラベル偏り（1000 ラウンド） | 表 13 | 12 | 40 時間以上 |

**時間は実測ではなく見込みです**（学習するサンプル数と、1 本あたり 300 サンプル/秒・4〜6 本同時という仮定から）。ステップ 1 のあと、出力の「s/round」で実際の速さがわかります。`STEPS` から不要な行を消せば、そのステップを飛ばせます。

**表 1 の大きな 4 タスク**：論文どおりの全グリッド（8 通り × 100 ラウンド、1 ラウンド 1 エポック）は約 15 億サンプルで現実的でないため、設定がすべて書かれている表 7（学習率 1e-3・バッチ 32・20 ラウンド。ランク 5 は表 1 の FedTT と同じ手法）で比べます。

## 設定とセットアップ
必要なら `USE_DRIVE`・`HOURS` だけ変えてください。

In [ ]:
import os, sys, time, subprocess, torch
from google.colab import drive

USE_DRIVE = True                                     # False: 結果を /content に置く（切断すると消える）
OUT   = '/content/drive/MyDrive/FedTT_repro/results' if USE_DRIVE else '/content/results'
HOURS = 23                                         # このセッションで使う時間（プランの上限より少し短く）
CODE  = '/content/fedtt_code'

if USE_DRIVE:
    drive.mount('/content/drive')
os.makedirs(OUT, exist_ok=True)
os.makedirs(f'{CODE}/fedtt', exist_ok=True)
os.makedirs(f'{CODE}/fedtt_repro', exist_ok=True)
os.environ['HF_HOME'] = '/content/hf_cache'          # モデル・データのダウンロード先（ローカルディスク）
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
assert torch.cuda.is_available(), 'GPU がありません。「ランタイム → ランタイムのタイプを変更」で GPU（A100 / L4 など）を選んでください'
gpu = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
bf16 = torch.cuda.is_bf16_supported()
cpus = os.cpu_count() or 2
PARALLEL = max(1, min(int((gpu_gb - 4) // 6), max(1, cpus // 2), 6))   # 1 つの GPU で同時に流すラン数
T_END = time.time() + 3600 * HOURS
print(f'GPU: {gpu}（{gpu_gb:.0f} GB）, CPU {cpus} 個 -> 同時実行 {PARALLEL} 本')
print('数値の精度:', 'bf16 が使える GPU（ただしモデルが bf16 で動かないときは、ランごとに自動で fp32/TF32 に切り替え）' if bf16
      else 'fp32（bf16 非対応の GPU。数倍遅くなります）')
print('結果の保存先:', OUT, '| コードの版:', 'bf46bcaab9')
subprocess.run('pip -q install "datasets>=2.18"', shell=True, check=False)
# Colab に入っている古い torchao（0.10）があると、新しい peft が LoRA 層を作るときに止まる（パラメータ数の照合の LoRA の行）。
# FedTT / FedTT+ は torchao も peft も使わないので、外すだけでよい。
subprocess.run('pip -q uninstall -y torchao', shell=True, check=False)

## コード（変更不要）
論文の擬似コードから書いた実装です。リポジトリ paper-reproductions の `papers/2025-findings-acl-fedtt/` にあるものと同じで、`fedtt/` が論文の手法（FedTT・FedTT+）、`fedtt_repro/` が再現実験の枠組みです。下のセルが `/content/fedtt_code/` に書き出します。

In [ ]:
%%writefile /content/fedtt_code/fedtt/__init__.py
"""FedTT / FedTT+ (Ghiasvand et al., Findings of ACL 2025): the method proposed in the paper.

  tt.py         tensor-train linear layer (Eq. 1) and the FedTT+ core schedule (Alg. 2, line 3)
  model.py      tensorized adapters and tensorized classifier inserted into a Hugging Face encoder,
                FedTT+ freezing per round (and LoRA-family references for side-by-side checks)
  algorithm.py  one communication round of Algorithm 1 (FedTT) / Algorithm 2 (FedTT+): local AdamW updates
                on each selected client and server-side averaging of exactly the uploaded parameters

The experiment code (GLUE data, client splits, suites, Colab notebooks) is in ../../reproduction.
"""
from .algorithm import local_batches, run_round, select_clients
from .model import FedModel, TensorizedAdapter, build_model
from .tt import TTLinear, plus_core_index, split_modes

__version__ = "0.1.0"


In [ ]:
%%writefile /content/fedtt_code/fedtt/tt.py
"""Tensor-train (TT) linear layer of FedTT (Ghiasvand et al., Findings of ACL 2025, Sec. 3.2 and 4.1).

A weight W in R^{P x Q} is reshaped into a tensor in R^{k_1 x ... x k_J} (prod k_j = P*Q) and stored as
J cores G_j in R^{r_{j-1} x k_j x r_j} with r_0 = r_J = 1 (paper Eq. 1):

    W = G_1 x_{3,1} G_2 x_{3,1} ... x_{3,1} G_J .

The first modes k_1..k_m multiply to P (input side), the rest to Q (output side); e.g. Table 10 gives
[8, 8, 12, 8, 8] for 768 x 64 (input 8*8*12, output 8*8) and for 64 x 768 (input 8*8, output 12*8*8).

Forward ("contract", the paper's way): the input activations are contracted with the cores one by one and
W is never formed ("the tensor factors are directly contracted with the vector of activation values").
"dense" builds W from the cores first and calls one matmul; both compute the same function and gradients
(checked in tests/test_core.py) and differ only in speed.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


def split_modes(shape, in_features, out_features):
    """Split a TT shape into (input modes, output modes): the shortest prefix whose product is P."""
    p = 1
    for m, k in enumerate(shape):
        p *= int(k)
        if p == in_features:
            in_modes, out_modes = tuple(shape[:m + 1]), tuple(shape[m + 1:])
            if math.prod(out_modes) != out_features:
                break
            return in_modes, out_modes
    raise ValueError(f"TT shape {shape} does not factor {in_features} x {out_features}")


class TTLinear(nn.Module):
    """y = x W + b with W in TT format (rank r between all neighbouring cores)."""

    def __init__(self, in_features, out_features, shape, rank, bias=True, w_std=0.02, mode="contract"):
        super().__init__()
        self.in_features, self.out_features = int(in_features), int(out_features)
        self.shape = tuple(int(k) for k in shape)
        self.in_modes, self.out_modes = split_modes(self.shape, self.in_features, self.out_features)
        self.n_in, self.J = len(self.in_modes), len(self.shape)
        self.rank = int(rank)
        self.ranks = [1] + [self.rank] * (self.J - 1) + [1]
        self.cores = nn.ParameterList(
            [nn.Parameter(torch.empty(self.ranks[j], self.shape[j], self.ranks[j + 1])) for j in range(self.J)])
        self.bias = nn.Parameter(torch.zeros(self.out_features)) if bias else None
        self.mode = mode
        self.reset_parameters(w_std)

    # ------------------------------------------------------------------ init
    @torch.no_grad()
    def reset_parameters(self, w_std):
        """i.i.d. N(0, s^2) core entries with s chosen so that every entry of W has std `w_std`:
        W_ij is a sum of prod(r_1..r_{J-1}) products of J core entries, so Var = prod(r) * s^(2J).
        (The paper does not state its initialization; see README.)"""
        n_terms = math.prod(self.ranks[1:-1])
        s = (w_std ** 2 / n_terms) ** (1.0 / (2 * self.J))
        for c in self.cores:
            c.normal_(0.0, s)
        if self.bias is not None:
            self.bias.zero_()

    # ------------------------------------------------------------------ forward
    def weight(self):
        """Dense W (P x Q), rebuilt from the cores (used by mode='dense' and by the tests)."""
        W = self.cores[0].reshape(-1, self.ranks[1])                     # (k_1, r_1)
        for j in range(1, self.J):
            G = self.cores[j]
            W = (W @ G.reshape(self.ranks[j], -1)).reshape(-1, self.ranks[j + 1])
        return W.reshape(self.in_features, self.out_features)

    def _contract(self, x2):
        B = x2.shape[0]
        t = x2.reshape(B, 1, self.in_features)                          # (B, r_0, rest)
        rest = self.in_features
        for j in range(self.n_in):                                       # absorb input modes k_1..k_m
            G = self.cores[j]
            k = G.shape[1]
            rest //= k
            t = torch.einsum("bakn,aks->bsn", t.reshape(B, G.shape[0], k, rest), G)
        t = t.reshape(B, 1, self.ranks[self.n_in])                       # (B, 1, r_m)
        for j in range(self.n_in, self.J):                               # emit output modes
            G = self.cores[j]
            t = torch.einsum("bpa,aks->bpks", t, G).reshape(B, -1, G.shape[2])
        return t.reshape(B, self.out_features)

    def forward(self, x):
        lead = x.shape[:-1]
        x2 = x.reshape(-1, self.in_features)
        if self.mode == "dense":
            y = F.linear(x2, self.weight().t().to(x2.dtype))
        else:
            y = self._contract(x2)
        if self.bias is not None:
            y = y + self.bias.to(y.dtype)
        return y.reshape(*lead, self.out_features)

    # ------------------------------------------------------------------ FedTT+
    def set_trainable_cores(self, idx, train_bias):
        """Only cores with 0-based index in `idx` get gradients (Alg. 2, lines 5-6)."""
        idx = set(idx)
        for j, c in enumerate(self.cores):
            c.requires_grad_(j in idx)
        if self.bias is not None:
            self.bias.requires_grad_(bool(train_bias))

    def n_core_params(self):
        return sum(c.numel() for c in self.cores)

    def extra_repr(self):
        return (f"{self.in_features}x{self.out_features}, shape={list(self.shape)} "
                f"(in {list(self.in_modes)} / out {list(self.out_modes)}), rank={self.rank}, "
                f"core params={self.n_core_params()}, bias={self.bias is not None}, mode={self.mode}")


def plus_core_index(t, J, schedule="text"):
    """FedTT+ (Alg. 2, line 3): the middle core trained in round t (1-based round; returns a 1-based index).

    'text'    r in {2, ..., J-1}, cycling 2, 3, ..., J-1, 2, ... (Sec. 4.3: "an index r is selected from the
              set r in {2, ..., J-1}", with G_1 and G_J always trained).
    'literal' the pseudocode as printed, mod(t, J-2) = r-1, i.e. r = (t mod (J-2)) + 1 in {1, ..., J-2};
              it never trains G_{J-1} and in some rounds trains only {G_1, G_J}. Kept for comparison.
    """
    if J < 3:
        raise ValueError("FedTT+ needs at least three cores")
    if schedule == "text":
        return 2 + (t - 1) % (J - 2)
    if schedule == "literal":
        return (t % (J - 2)) + 1
    raise ValueError(schedule)


In [ ]:
%%writefile /content/fedtt_code/fedtt/model.py
"""Tensorized adapters and the tensorized classifier of FedTT (Sec. 4.1, Fig. 1, Table 10), injected into a
Hugging Face encoder; optional LoRA-family references (PEFT) for side-by-side checks.

Placement (Fig. 1b): one adapter after the multi-head attention and one after the MLP of every selected
block, i.e. on the outputs of `attention.output.dense` and `output.dense` (before dropout, residual and
LayerNorm, as in Houlsby et al. 2019). Adapter: h -> h + up(act(down(h))), down: H -> 64, up: 64 -> H,
both TT layers (Table 10 shapes, rank 5 by default). Classification (Fig. 1c): the H x H dense layer of the
classification head is replaced by a TT layer; the final H x C projection stays dense and trainable.
The backbone (including LayerNorms) is frozen.
"""
import math
import re

import torch
import torch.nn as nn

from .tt import TTLinear, plus_core_index

# (in, out) -> TT shape. 768/4096 rows are Table 10 of the paper. The paper gives no shapes for hidden size
# 1024 (RoBERTa-large); the 1024 rows are our analogue of the 768 rows (8*8*16 instead of 8*8*12) and are
# checked against the parameter counts of Table 12 by `python -m fedtt_repro.count`.
SHAPES = {
    (768, 64): (8, 8, 12, 8, 8),
    (64, 768): (8, 8, 12, 8, 8),
    (4096, 64): (16, 16, 16, 4, 4, 4),
    (64, 4096): (4, 4, 4, 16, 16, 16),
    (768, 768): (12, 8, 8, 8, 8, 12),
    (1024, 64): (8, 8, 16, 8, 8),
    (64, 1024): (8, 8, 16, 8, 8),
    (1024, 1024): (16, 8, 8, 8, 8, 16),
}

TT_METHODS = ("fedtt", "fedttplus")
LORA_METHODS = ("lora", "ffa", "rolora")


class TensorizedAdapter(nn.Module):
    def __init__(self, hidden, bottleneck, rank, act="relu", w_std=0.01, bias=True, mode="contract",
                 shapes=None):
        super().__init__()
        shapes = shapes or SHAPES
        self.down = TTLinear(hidden, bottleneck, shapes[(hidden, bottleneck)], rank, bias, w_std, mode)
        self.up = TTLinear(bottleneck, hidden, shapes[(bottleneck, hidden)], rank, bias, w_std, mode)
        self.act = {"relu": nn.ReLU(), "gelu": nn.GELU(), "tanh": nn.Tanh(), "silu": nn.SiLU()}[act]

    def forward(self, h):
        return h + self.up(self.act(self.down(h)))


class WithAdapter(nn.Module):
    """x -> adapter(base(x)); `base` is the frozen pretrained projection."""

    def __init__(self, base, adapter):
        super().__init__()
        self.base, self.adapter = base, adapter

    def forward(self, x):
        return self.adapter(self.base(x))


_SITE = re.compile(r"(^|\.)encoder\.layer\.(\d+)\.(attention\.output|output)\.dense$")


def _set_module(root, name, new):
    *path, leaf = name.split(".")
    parent = root
    for p in path:
        parent = getattr(parent, p)
    setattr(parent, leaf, new)


def select_blocks(n_layers, spec):
    """'all' | 'lastN' | comma list of block indices."""
    if spec in (None, "all"):
        return list(range(n_layers))
    if isinstance(spec, str) and spec.startswith("last"):
        n = int(spec[4:])
        return list(range(n_layers - n, n_layers))
    if isinstance(spec, str):
        return [int(s) for s in spec.split(",")]
    return list(spec)


def adapter_sites(model, blocks):
    names = []
    for n, m in model.named_modules():
        hit = _SITE.search(n)
        if hit and isinstance(m, nn.Linear) and int(hit.group(2)) in blocks:
            names.append(n)
    if len(names) != 2 * len(blocks):
        raise RuntimeError(f"expected {2 * len(blocks)} adapter sites, found {names[:4]}...")
    return names


def head_modules(model):
    """(name of the H x H dense layer of the classification head, name of the H x C output layer)."""
    if hasattr(model, "classifier") and hasattr(model.classifier, "dense"):      # RoBERTa
        return "classifier.dense", "classifier.out_proj"
    if hasattr(model, "pooler") and hasattr(model.pooler, "dense"):              # DeBERTa (v1/v2)
        return "pooler.dense", "classifier"
    raise RuntimeError(f"unknown classification head for {type(model).__name__}")


def _lora_targets(model):
    t = model.config.model_type
    if t == "roberta" or t == "bert":
        return ["query", "value"]
    if t == "deberta":
        return ["in_proj"]                    # fused q/k/v projection of DeBERTa v1
    if t == "deberta-v2":
        return ["query_proj", "value_proj"]
    raise RuntimeError(f"no LoRA targets for model type {t}")


class FedModel(nn.Module):
    """Wraps the HF model and implements the per-round trainable sets of each method."""

    def __init__(self, hf, cfg, tt_adapter_layers, head_names, global_names):
        super().__init__()
        self.hf, self.cfg = hf, cfg
        self.tt_adapter_layers = tt_adapter_layers     # TT layers subject to the FedTT+ freezing schedule
        self.head_names = head_names
        self.global_names = global_names               # parameters kept by the server (trainable in some round)

    def forward(self, input_ids, attention_mask):
        return self.hf(input_ids=input_ids, attention_mask=attention_mask).logits

    def set_round(self, t):
        """Set requires_grad for round t (1-based); returns a short description of the round's schedule."""
        m = self.cfg["method"]
        if m == "fedttplus":
            sched = self.cfg.get("plus_schedule", "text")
            rs = set()
            for L in self.tt_adapter_layers:
                r = plus_core_index(t, L.J, sched)                     # 1-based core index (Alg. 2, line 3)
                L.set_trainable_cores({0, r - 1, L.J - 1}, self.cfg.get("plus_train_bias", False))
                rs.add(r)
            return f"r={'/'.join(map(str, sorted(rs)))}"
        if m == "rolora":                                              # B in odd rounds, A in even rounds
            for n, p in self.hf.named_parameters():
                if "lora_B" in n:
                    p.requires_grad_(t % 2 == 1)
                elif "lora_A" in n:
                    p.requires_grad_(t % 2 == 0)
            return "B" if t % 2 == 1 else "A"
        return ""

    def trainable(self):
        return [(n, p) for n, p in self.hf.named_parameters() if p.requires_grad]


def tiny_config(model_type, vocab_size, n_layers=2, hidden=768, num_labels=2):
    """Small random encoders with a supported hidden size (tests and offline counting only)."""
    from transformers import DebertaConfig, RobertaConfig
    kw = dict(vocab_size=vocab_size, hidden_size=hidden, num_hidden_layers=n_layers, num_attention_heads=4,
              intermediate_size=256, hidden_dropout_prob=0.0, attention_probs_dropout_prob=0.0,
              num_labels=num_labels)
    if model_type == "deberta":
        return DebertaConfig(max_position_embeddings=136, **kw)
    return RobertaConfig(max_position_embeddings=136, **kw)


def build_model(cfg, num_labels, hf_config=None, pretrained=True):
    """cfg keys: model, method, rank, adapter_layers, bottleneck, act, adapter_init_std, tt_mode, tt_classifier,
    lora_alpha. `hf_config` (with pretrained=False) builds an untrained model from a config (counting/tests)."""
    from transformers import AutoModelForSequenceClassification
    torch.manual_seed(int(cfg.get("seed", 0)))
    if hf_config is not None and not pretrained:
        hf = AutoModelForSequenceClassification.from_config(hf_config)
    else:
        hf = AutoModelForSequenceClassification.from_pretrained(cfg["model"], num_labels=num_labels)
    for p in hf.parameters():
        p.requires_grad_(False)
    method = cfg["method"]
    H = hf.config.hidden_size
    n_layers = hf.config.num_hidden_layers
    blocks = select_blocks(n_layers, cfg.get("adapter_layers", "all"))
    head_dense, head_out = head_modules(hf)
    tt_layers = []
    rank = int(cfg.get("rank", 5))
    if method in TT_METHODS:
        for name in adapter_sites(hf, blocks):
            ad = TensorizedAdapter(H, int(cfg.get("bottleneck", 64)), rank, cfg.get("act", "relu"),
                                   float(cfg.get("adapter_init_std", 0.01)), True, cfg.get("tt_mode", "contract"))
            _set_module(hf, name, WithAdapter(hf.get_submodule(name), ad))
            tt_layers += [ad.down, ad.up]
            for p in ad.parameters():
                p.requires_grad_(True)
        if cfg.get("tt_classifier", True):
            std = getattr(hf.config, "initializer_range", 0.02)
            _set_module(hf, head_dense, TTLinear(H, H, SHAPES[(H, H)], rank, True, std, cfg.get("tt_mode", "contract")))
        for name in (head_dense, head_out):
            for p in hf.get_submodule(name).parameters():
                p.requires_grad_(True)
    elif method in LORA_METHODS:
        from peft import LoraConfig, get_peft_model
        kw = dict(r=rank, lora_alpha=float(cfg.get("lora_alpha", 2 * rank)), lora_dropout=0.0, bias="none",
                  target_modules=_lora_targets(hf), modules_to_save=[head_dense.split(".")[0], head_out.split(".")[0]])
        if cfg.get("adapter_layers", "all") != "all":
            kw.update(layers_to_transform=blocks, layers_pattern="layer")
        hf = get_peft_model(hf, LoraConfig(**kw))
        if method == "ffa":                    # FFA-LoRA: A frozen at its (shared) random init
            for n, p in hf.named_parameters():
                if "lora_A" in n:
                    p.requires_grad_(False)
    elif method == "head":
        for name in (head_dense, head_out):
            for p in hf.get_submodule(name).parameters():
                p.requires_grad_(True)
    else:
        raise ValueError(f"unknown method {method}")
    # Before the first set_round() everything that is trainable in some round still has requires_grad=True
    # (FedTT+ cores and biases, RoLoRA A and B); FFA-LoRA's A is frozen for good and stays out.
    global_names = [n for n, p in hf.named_parameters() if p.requires_grad]
    head_names = [n for n, _ in hf.named_parameters() if n.startswith(head_dense) or n.startswith(head_out)
                  or ".modules_to_save." in n]
    return FedModel(hf, cfg, tt_layers, head_names, global_names)


def count_round_params(fm, t):
    """Parameters a client uploads in round t (= trainable parameters of that round)."""
    fm.set_round(t)
    return sum(p.numel() for _, p in fm.trainable())


def param_breakdown(fm, t=1):
    fm.set_round(t)
    out = {"adapter_cores": 0, "adapter_bias": 0, "head": 0, "lora": 0, "other": 0}
    for n, p in fm.trainable():
        if ".adapter." in n and ".cores." in n:
            out["adapter_cores"] += p.numel()
        elif ".adapter." in n and n.endswith("bias"):
            out["adapter_bias"] += p.numel()
        elif n in fm.head_names or ".modules_to_save." in n:
            out["head"] += p.numel()
        elif "lora_" in n:
            out["lora"] += p.numel()
        else:
            out["other"] += p.numel()
    return out


In [ ]:
%%writefile /content/fedtt_code/fedtt/algorithm.py
"""One communication round of Algorithm 1 (FedTT) and Algorithm 2 (FedTT+), and client selection.

Per communication round t (1-based):
  FedTT  (Alg. 1)  every selected client starts from the global trainable parameters w^(t), performs K local
                   updates (AdamW) on its data, and sends w_i^(t)+K; the server sets w^(t+1) = mean_i w_i^(t)+K
                   and broadcasts it.
  FedTT+ (Alg. 2)  the same, but in each tensorized adapter layer only {G_1, G_r, G_J} are trainable
                   (r from tt.plus_core_index, set by FedModel.set_round), the other cores stay frozen and
                   identical on all clients; clients send and the server averages and broadcasts only
                   {G_1, G_r, G_J} (plus the classification head, which is always trainable).
Cross-silo: all N clients take part in every round. Large-scale cross-device (LSCD): `clients_per_round`
clients are drawn uniformly without replacement each round.

One model instance is reused for all clients (sequential simulation); clients are stateless (a fresh AdamW
per client and round), which is what the pseudocode describes. A client's data is any object with len() and
batch(positions, pad_id) -> (input_ids, attention_mask, labels) as numpy arrays.
"""
import math

import numpy as np
import torch
import torch.nn.functional as F


def local_batches(n_local, cfg, t, c):
    """Positions (relative to the client's data) of the K minibatches of client c in round t."""
    rng = np.random.default_rng([int(cfg["seed"]), int(t), int(c), 7])
    bs = int(cfg["bs"])
    if cfg.get("local_steps"):
        K = int(cfg["local_steps"])
        need = K * bs
        order = np.concatenate([rng.permutation(n_local) for _ in range(math.ceil(need / n_local))])[:need]
        return [order[i * bs:(i + 1) * bs] for i in range(K)]
    E = float(cfg["local_epochs"])
    order = np.concatenate([rng.permutation(n_local) for _ in range(math.ceil(E))])[:int(round(E * n_local))]
    return [order[i:i + bs] for i in range(0, len(order), bs)]


def run_round(fm, gstate, clients, sel, cfg, t, device, pad_id, amp_dtype):
    """One communication round of Alg. 1 / Alg. 2. Updates `gstate` in place and returns round statistics.

    Every selected client loads the global parameters, runs its K local AdamW updates on the parameters
    that are trainable in round t, and uploads exactly those; the server averages them (uniform 1/|S| as
    in the pseudocode, or weighted by client size with cfg agg='weighted') and keeps all other entries.
    """
    params = dict(fm.hf.named_parameters())
    sched = fm.set_round(t)
    tr = fm.trainable()
    tr_names = [n for n, _ in tr]

    def load_global():
        with torch.no_grad():
            for n, v in gstate.items():
                params[n].copy_(v)

    acc = {n: torch.zeros_like(gstate[n], dtype=torch.float32) for n in tr_names}
    wsum, loss_sum, n_steps = 0.0, torch.zeros((), device=device), 0
    fm.train()
    for c in sel:
        load_global()
        opt = torch.optim.AdamW([p for _, p in tr], lr=float(cfg["lr"]), weight_decay=float(cfg["weight_decay"]))
        cd = clients[c]
        micro = int(cfg.get("micro_bs") or 0)
        for rel in local_batches(len(cd), cfg, t, c):
            opt.zero_grad(set_to_none=True)
            chunks = [rel] if not micro or len(rel) <= micro else [rel[i:i + micro] for i in range(0, len(rel), micro)]
            loss = torch.zeros((), device=device)
            for ch in chunks:            # gradient accumulation: same mean-loss gradient as the full minibatch
                x, msk, y = cd.batch(ch, pad_id)
                x, msk, y = (torch.from_numpy(a).to(device, non_blocking=True) for a in (x, msk, y))
                with torch.autocast(device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
                    logits = fm(x, msk)
                part = F.cross_entropy(logits.float(), y) * (len(ch) / len(rel))
                part.backward()
                loss = loss + part.detach()
            if cfg["max_grad_norm"]:
                torch.nn.utils.clip_grad_norm_([p for _, p in tr], float(cfg["max_grad_norm"]))
            opt.step()
            loss_sum += loss.detach()
            n_steps += 1
        w = float(len(cd)) if cfg["agg"] == "weighted" else 1.0
        with torch.no_grad():
            for n in tr_names:
                acc[n] += w * params[n].detach().float()
        wsum += w
    with torch.no_grad():                        # server: average what was uploaded, keep the rest
        for n in tr_names:
            gstate[n] = (acc[n] / wsum).to(gstate[n].dtype)
    load_global()
    return dict(sched=sched, loss=float(loss_sum) / max(n_steps, 1), steps=n_steps, clients=len(sel),
                up_params=int(sum(p.numel() for _, p in tr)), uploaded=tr_names)


def select_clients(cfg, t):
    N, m = int(cfg["n_clients"]), cfg.get("clients_per_round")
    if m and int(m) < N:                           # LSCD: m clients drawn uniformly each round
        return np.sort(np.random.default_rng([int(cfg["seed"]), int(t), 3]).choice(N, int(m), replace=False))
    return np.arange(N)                            # cross-silo: everyone


In [ ]:
%%writefile /content/fedtt_code/fedtt_repro/__init__.py
"""Reproduction experiments for FedTT / FedTT+ (Ghiasvand et al., Findings of ACL 2025).

The method itself is the package `fedtt` in ../../method (on Colab both packages sit side by side in
/content/fedtt_code). This makes `import fedtt` work when running from the reproduction folder.
"""
import os
import sys

_METHOD = os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__)))), "method")
if os.path.isdir(os.path.join(_METHOD, "fedtt")) and _METHOD not in sys.path:
    sys.path.insert(0, _METHOD)

__version__ = "0.1.0"


In [ ]:
%%writefile /content/fedtt_code/fedtt_repro/data.py
"""GLUE data, the FedPETuning train/dev split used by the paper (Table 8) and the client partitions.

Splits (FedPETuning, Zhang et al. 2023, which the paper follows; Table 8 train sizes):
  'train'   GLUE train minus a held-out part, so that |train| matches Table 8
            (MRPC 3,301 / SST-2 66,675 / QNLI 103,695 / QQP 360,210 / MNLI 388,774; RTE 2,241);
  'heldout' the rest of GLUE train (FedPETuning's dev set; 367 / 674 / 1,048 / 3,636 / 3,928 / 249);
  'val'     the GLUE validation set (MNLI: validation_matched) -- the set whose best score over rounds
            the paper reports ("we record the best validation results"). For DeBERTa on QQP the paper
            uses 1,000 random validation samples (cfg val_subsample=1000).
The split uses a fixed seed (42), independent of the run seed, so every run sees the same data.
"""
import os

import numpy as np

TASKS = {
    "mrpc": dict(keys=("sentence1", "sentence2"), n_cls=2, metric="f1", train_size=3301, val="validation"),
    "sst2": dict(keys=("sentence", None), n_cls=2, metric="acc", train_size=66675, val="validation"),
    "qnli": dict(keys=("question", "sentence"), n_cls=2, metric="acc", train_size=103695, val="validation"),
    "qqp": dict(keys=("question1", "question2"), n_cls=2, metric="acc", train_size=360210, val="validation"),
    "mnli": dict(keys=("premise", "hypothesis"), n_cls=3, metric="acc", train_size=388774,
                 val="validation_matched"),
    "rte": dict(keys=("sentence1", "sentence2"), n_cls=2, metric="acc", train_size=2241, val="validation"),
}

# Per-client label distributions (rows = clients) for the 3-client label-skew settings of Appendix B /
# Table 13 (copied from FFA-LoRA). 'severe' is the FedTT text ([0.05, 0.95]); FFA-LoRA's own paper uses
# [0.1, 0.9] for binary tasks, available as 'severe_ffa'.
RATIOS = {
    "mild": {2: [[0.15, 0.85], [0.85, 0.15], [0.5, 0.5]],
             3: [[0.6, 0.2, 0.2], [0.2, 0.6, 0.2], [0.2, 0.2, 0.6]]},
    "severe": {2: [[0.05, 0.95], [0.95, 0.05], [0.5, 0.5]],
               3: [[0.9, 0.05, 0.05], [0.05, 0.9, 0.05], [0.05, 0.05, 0.9]]},
    "severe_ffa": {2: [[0.1, 0.9], [0.9, 0.1], [0.5, 0.5]],
                   3: [[0.9, 0.05, 0.05], [0.05, 0.9, 0.05], [0.05, 0.05, 0.9]]},
}


class TokData:
    """Tokenized examples stored as one flat int32 array plus offsets; `idx` selects a subset."""

    def __init__(self, ids, off, labels, idx=None):
        self.ids, self.off, self.labels_all = ids, off, labels
        self.idx = np.arange(len(labels)) if idx is None else np.asarray(idx)

    def __len__(self):
        return len(self.idx)

    @property
    def labels(self):
        return self.labels_all[self.idx]

    def lengths(self):
        return (self.off[self.idx + 1] - self.off[self.idx]).astype(np.int64)

    def subset(self, rel):
        """Subset by positions relative to this view."""
        return TokData(self.ids, self.off, self.labels_all, self.idx[np.asarray(rel)])

    def batch(self, rel, pad_id):
        """Padded (input_ids, attention_mask, labels) as numpy arrays for positions `rel` of this view."""
        j = self.idx[np.asarray(rel)]
        lens = self.off[j + 1] - self.off[j]
        L = int(lens.max())
        x = np.full((len(j), L), pad_id, dtype=np.int64)
        m = np.zeros((len(j), L), dtype=np.int64)
        for i, (a, b) in enumerate(zip(self.off[j], self.off[j + 1])):
            x[i, :b - a] = self.ids[a:b]
            m[i, :b - a] = 1
        return x, m, self.labels_all[j].astype(np.int64)


def _slug(name):
    return name.replace("/", "__")


def _tokenize(texts_a, texts_b, tok, max_len, chunk=20000):
    ids, lens = [], []
    for s in range(0, len(texts_a), chunk):
        a = texts_a[s:s + chunk]
        b = None if texts_b is None else texts_b[s:s + chunk]
        enc = tok(a, b, truncation=True, max_length=max_len)["input_ids"]
        for e in enc:
            ids.append(np.asarray(e, dtype=np.int32))
            lens.append(len(e))
    off = np.zeros(len(lens) + 1, dtype=np.int64)
    np.cumsum(lens, out=off[1:])
    return np.concatenate(ids) if ids else np.zeros(0, np.int32), off


def load_glue(task, tok, tok_name, max_len, cache_dir):
    """Returns {'train_full': TokData, 'val': TokData} (cached as .npz under cache_dir)."""
    spec = TASKS[task]
    os.makedirs(cache_dir, exist_ok=True)
    path = os.path.join(cache_dir, f"{_slug(tok_name)}_{task}_L{max_len}.npz")
    if os.path.exists(path):
        z = np.load(path)
        return {s: TokData(z[f"{s}_ids"], z[f"{s}_off"], z[f"{s}_y"]) for s in ("train_full", "val")}
    from datasets import load_dataset
    ds = load_dataset("nyu-mll/glue", task)
    out, arrays = {}, {}
    for s, split in (("train_full", "train"), ("val", spec["val"])):
        d = ds[split]
        ka, kb = spec["keys"]
        ids, off = _tokenize(list(d[ka]), None if kb is None else list(d[kb]), tok, max_len)
        y = np.asarray(d["label"], dtype=np.int64)
        arrays.update({f"{s}_ids": ids, f"{s}_off": off, f"{s}_y": y})
        out[s] = TokData(ids, off, y)
    tmp = path + f".tmp{os.getpid()}.npz"
    np.savez(tmp, **arrays)
    _replace(tmp, path)
    return out


def _replace(src, dst, tries=20):
    import time
    for k in range(tries):          # OneDrive / Google Drive may hold a lock for a moment
        try:
            os.replace(src, dst)
            return
        except PermissionError:
            time.sleep(0.5 * (k + 1))
    os.replace(src, dst)


def fedpet_split(task, train_full, split_seed=42):
    """(train, heldout) with |train| = Table 8 size, rest held out (FedPETuning)."""
    n = len(train_full)
    n_train = min(TASKS[task]["train_size"], n)
    perm = np.random.default_rng(split_seed).permutation(n)
    return train_full.subset(perm[n - n_train:]), train_full.subset(perm[:n - n_train])


def subsample(data, k, seed=42):
    if k is None or k >= len(data):
        return data
    return data.subset(np.sort(np.random.default_rng(seed).choice(len(data), int(k), replace=False)))


# ---------------------------------------------------------------------------------------------- partitions
def partition(labels, n_clients, kind, rng, alpha=1.0, n_cls=None, min_size=None):
    """List of index arrays (positions in `labels`), one per client.

    'iid'                 uniform random split into equal parts;
    'dir'                 label skew, per class p_c ~ Dir(alpha * 1_N) over clients with the NIID-Bench / FedLab
                          balancing rule, redrawn until every client has >= min_size samples (FedLab's
                          hetero_dir_partition default: min_size = number of classes; FedPETuning uses FedLab
                          with alpha = 1.0);
    'mild' | 'severe' | 'severe_ffa'   3-client label-ratio splits of Appendix B (RATIOS above).
    """
    labels = np.asarray(labels)
    n = len(labels)
    if kind == "iid":
        return [np.sort(p) for p in np.array_split(rng.permutation(n), n_clients)]
    classes = np.unique(labels) if n_cls is None else np.arange(n_cls)
    if kind == "dir":
        min_size = len(classes) if min_size is None else min_size
        for _ in range(1000):
            batches = [[] for _ in range(n_clients)]
            for c in classes:
                idx_c = np.flatnonzero(labels == c)
                rng.shuffle(idx_c)
                p = rng.dirichlet(alpha * np.ones(n_clients))
                sizes = np.array([len(b) for b in batches])
                p = p * (sizes < n / n_clients)
                p = p / p.sum()
                cuts = (np.cumsum(p) * len(idx_c)).astype(int)[:-1]
                for b, part in zip(batches, np.split(idx_c, cuts)):
                    b.extend(part.tolist())
            if min(len(b) for b in batches) >= min_size:
                return [np.sort(np.asarray(b, dtype=np.int64)) for b in batches]
        raise RuntimeError("Dirichlet partition: could not reach the minimum client size")
    if kind in RATIOS:
        P = np.asarray(RATIOS[kind][len(classes)], dtype=float)
        if P.shape[0] != n_clients:
            raise ValueError(f"'{kind}' is defined for {P.shape[0]} clients")
        out = [[] for _ in range(n_clients)]
        for ci, c in enumerate(classes):
            idx_c = np.flatnonzero(labels == c)
            rng.shuffle(idx_c)
            w = P[:, ci] / P[:, ci].sum()      # share of class c sent to each client
            cuts = np.round(np.cumsum(w) * len(idx_c)).astype(int)[:-1]
            for k, part in enumerate(np.split(idx_c, cuts)):
                out[k].extend(part.tolist())
        return [np.sort(np.asarray(b, dtype=np.int64)) for b in out]
    raise ValueError(kind)


# ---------------------------------------------------------------------------------------------- metrics
def metrics(pred, y):
    pred, y = np.asarray(pred), np.asarray(y)
    acc = float((pred == y).mean()) * 100
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y != 1)).sum())
    fn = int(((pred != 1) & (y == 1)).sum())
    f1 = 100.0 * 2 * tp / max(2 * tp + fp + fn, 1)
    return {"acc": acc, "f1": f1}


In [ ]:
%%writefile /content/fedtt_code/fedtt_repro/fl.py
"""Experiment runner for Algorithm 1 (FedTT) and Algorithm 2 (FedTT+): data, rounds, evaluation, logs.

The per-round logic (local updates and server averaging) is in method/fedtt/algorithm.py; this module
feeds it GLUE data, evaluates every round, writes log.jsonl / result.json and resumes from checkpoints.

Per communication round t (1-based):
  FedTT  (Alg. 1)  every selected client starts from the global trainable parameters w^(t), performs K local
                   updates (AdamW) on its data, and sends w_i^(t)+K; the server sets w^(t+1) = mean_i w_i^(t)+K
                   and broadcasts it.
  FedTT+ (Alg. 2)  the same, but in each tensorized adapter layer only {G_1, G_r, G_J} are trainable
                   (r from tt.plus_core_index), the other cores stay frozen and identical on all clients;
                   clients send and the server averages and broadcasts only {G_1, G_r, G_J} (plus the
                   classification head, which is always trainable).
Cross-silo: all N clients take part in every round. Large-scale cross-device (LSCD): `clients_per_round`
clients are drawn uniformly without replacement each round.

One model instance is reused for all clients (sequential simulation); clients are stateless (a fresh AdamW
per client and round), which is what the pseudocode describes.
"""
import hashlib
import json
import math
import os
import platform
import sys
import time

import numpy as np
import torch
import torch.nn.functional as F

from . import data as D
from fedtt.algorithm import local_batches, run_round, select_clients  # noqa: F401  (method/fedtt)
from fedtt.model import build_model

DEFAULTS = dict(
    suite="adhoc", task="mrpc", model="microsoft/deberta-base", method="fedtt", rank=5,
    adapter_layers="all", bottleneck=64, act="relu", adapter_init_std=0.01, tt_mode="contract",
    tt_classifier=True, plus_schedule="text", plus_train_bias=False, lora_alpha=None,
    n_clients=5, clients_per_round=None, partition="iid", alpha=1.0,
    rounds=100, local_epochs=1.0, local_steps=None, bs=32, lr=1e-3, weight_decay=0.0, max_grad_norm=0.0,
    agg="uniform", max_len=128, eval_every=1, eval_bs=128, eval_heldout=True,
    val_subsample=None, heldout_subsample=None, train_subsample=None,
    seed=0, amp="auto", ckpt_every=10, threads=None, micro_bs=None, tag="",
)
# Fields shown in the run id; any other field that differs from DEFAULTS adds a short hash.
_ID_FIELDS = ("suite", "task", "model", "method", "rank", "adapter_layers", "n_clients", "clients_per_round",
              "partition", "alpha", "rounds", "local_epochs", "local_steps", "bs", "lr", "seed", "tag")
_NOT_IN_ID = ("eval_bs", "ckpt_every", "threads", "amp", "eval_every", "tt_mode", "micro_bs")


def full_cfg(cfg):
    out = dict(DEFAULTS)
    out.update({k: v for k, v in cfg.items() if v is not None or k in ("clients_per_round", "local_steps",
                                                                       "local_epochs")})
    if out.get("lora_alpha") is None and out["method"] in ("lora", "ffa", "rolora"):
        out["lora_alpha"] = 2 * out["rank"]
    return out


def run_id(cfg):
    c = full_cfg(cfg)
    m = c["model"].split("/")[-1]
    lay = "" if c["adapter_layers"] == "all" else f"-{c['adapter_layers']}"
    who = f"N{c['n_clients']}" + (f"m{c['clients_per_round']}" if c["clients_per_round"] else "")
    part = c["partition"] + (f"{c['alpha']:g}" if c["partition"] == "dir" else "")
    loc = f"K{c['local_steps']}" if c["local_steps"] else f"E{c['local_epochs']:g}"
    rid = (f"{c['task']}/{m}_{c['method']}-r{c['rank']}{lay}_{who}-{part}-{loc}-T{c['rounds']}"
           f"_lr{c['lr']:g}-bs{c['bs']}-s{c['seed']}")
    extra = {k: v for k, v in c.items() if k not in _ID_FIELDS and k not in _NOT_IN_ID and DEFAULTS.get(k) != v}
    if c["tag"]:
        rid += f"-{c['tag']}"
    if extra:
        rid += "-" + hashlib.sha1(json.dumps(extra, sort_keys=True, default=str).encode()).hexdigest()[:6]
    return rid


def run_dir(out_root, cfg):
    c = full_cfg(cfg)
    return os.path.join(out_root, "runs", c["suite"], *run_id(c).split("/"))


def env_info(device):
    d = dict(python=sys.version.split()[0], torch=torch.__version__, platform=platform.platform(),
             device=str(device))
    try:
        import transformers
        d["transformers"] = transformers.__version__
    except Exception:
        pass
    if device.type == "cuda":
        d["gpu"] = torch.cuda.get_device_name(0)
    else:
        d["cpu"] = platform.processor()
        d["threads"] = torch.get_num_threads()
    return d


def _write_json(path, obj):
    tmp = path + f".tmp{os.getpid()}"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=1, default=str)
    D._replace(tmp, path)


def _amp_dtype(mode, device):
    if mode == "fp32" or device.type != "cuda":
        return None
    if mode in ("auto", "bf16") and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return None


def probe_amp(fm, data, device, pad_id, amp_dtype):
    """Some Hugging Face models fail under bf16 autocast (DeBERTa: masked_fill with float32's minimum on bf16
    attention scores -> "value cannot be converted to type BFloat16 without overflow"). Run one small batch;
    if it fails, fall back to fp32 (TF32 tensor-core matmuls on Ampere and newer GPUs).
    Returns (dtype or None, description)."""
    if amp_dtype is None:
        return None, "fp32"
    x, m, _ = data.batch(np.arange(min(4, len(data))), pad_id)
    try:
        fm.eval()
        with torch.no_grad(), torch.autocast(device.type, dtype=amp_dtype):
            fm(torch.from_numpy(x).to(device), torch.from_numpy(m).to(device))
        return amp_dtype, str(amp_dtype).replace("torch.", "")
    except RuntimeError as e:
        return None, f"fp32 (tf32 matmuls); {str(amp_dtype).replace('torch.', '')} autocast failed: {str(e)[:100]}"


@torch.no_grad()
def evaluate(fm, data, bs, device, pad_id, amp_dtype):
    if data is None or len(data) == 0:
        return None
    fm.eval()
    order = np.argsort(data.lengths(), kind="stable")          # length-sorted batches (less padding)
    preds = np.empty(len(data), dtype=np.int64)
    for s in range(0, len(order), bs):
        rel = order[s:s + bs]
        x, m, _ = data.batch(rel, pad_id)
        with torch.autocast(device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            logits = fm(torch.from_numpy(x).to(device), torch.from_numpy(m).to(device))
        preds[rel] = logits.argmax(-1).cpu().numpy()
    return D.metrics(preds, data.labels)


def prepare_data(cfg, out_root, tok):
    task = cfg["task"]
    raw = D.load_glue(task, tok, cfg["model"], int(cfg["max_len"]), os.path.join(out_root, "cache"))
    train, heldout = D.fedpet_split(task, raw["train_full"])
    train = D.subsample(train, cfg.get("train_subsample"))
    heldout = D.subsample(heldout, cfg.get("heldout_subsample"))
    val = D.subsample(raw["val"], cfg.get("val_subsample"))
    return train, heldout, val


def run(cfg, out_root, deadline=None, verbose=True, device=None):
    """Runs (or resumes) one configuration; returns the result dict, or {'status': 'paused'} at the deadline."""
    cfg = full_cfg(cfg)
    rdir = run_dir(out_root, cfg)
    os.makedirs(rdir, exist_ok=True)
    res_path, ckpt_path = os.path.join(rdir, "result.json"), os.path.join(rdir, "ckpt.pt")
    if os.path.exists(res_path):
        with open(res_path, encoding="utf-8") as f:
            return json.load(f)
    if cfg.get("threads"):
        torch.set_num_threads(int(cfg["threads"]))
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    amp_dtype = _amp_dtype(cfg["amp"], device)

    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(cfg["model"])
    pad_id = tok.pad_token_id
    train, heldout, val = prepare_data(cfg, out_root, tok)
    spec = D.TASKS[cfg["task"]]
    parts = D.partition(train.labels, int(cfg["n_clients"]), cfg["partition"],
                        np.random.default_rng([int(cfg["seed"]), 11]), float(cfg["alpha"]), spec["n_cls"])
    clients = [train.subset(p) for p in parts]

    fm = build_model(cfg, spec["n_cls"]).to(device)
    amp_dtype, amp_desc = probe_amp(fm, val, device, pad_id, amp_dtype)
    if verbose:
        print(f"[{run_id(cfg)}] device {device}, precision {amp_desc}", flush=True)
    params = dict(fm.hf.named_parameters())
    gnames = list(fm.global_names)
    gstate = {n: params[n].detach().clone() for n in gnames}
    log, start = [], 1
    log_path = os.path.join(rdir, "log.jsonl")
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=device)
        gstate = {n: ck["gstate"][n].to(device) for n in gnames}
        log, start = ck["log"], ck["round"] + 1
        if verbose:
            print(f"[{run_id(cfg)}] resuming at round {start}", flush=True)
    with open(log_path, "w", encoding="utf-8") as f:       # drop rows after the last checkpoint
        f.writelines(json.dumps(r) + "\n" for r in log)

    def save_ckpt(t):
        tmp = ckpt_path + f".tmp{os.getpid()}"
        torch.save({"gstate": {n: v.detach().cpu() for n, v in gstate.items()}, "log": log, "round": t}, tmp)
        D._replace(tmp, ckpt_path)

    metric = spec["metric"]
    t0_run = time.time()
    with open(os.path.join(rdir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=1)
    for t in range(start, int(cfg["rounds"]) + 1):
        t0 = time.time()
        st = run_round(fm, gstate, clients, select_clients(cfg, t), cfg, t, device, pad_id, amp_dtype)
        sched, up_params = st["sched"], st["up_params"]
        train_sec = time.time() - t0
        row = dict(round=t, sched=sched, loss=st["loss"], steps=st["steps"], clients=st["clients"],
                   up_params=up_params, train_sec=round(train_sec, 2))
        if t % int(cfg["eval_every"]) == 0 or t == int(cfg["rounds"]):
            te = time.time()
            vm = evaluate(fm, val, int(cfg["eval_bs"]), device, pad_id, amp_dtype)
            hm = evaluate(fm, heldout, int(cfg["eval_bs"]), device, pad_id, amp_dtype) if cfg["eval_heldout"] else None
            row.update(val=vm[metric], val_acc=vm["acc"], val_f1=vm["f1"],
                       heldout=None if hm is None else hm[metric], eval_sec=round(time.time() - te, 2))
        log.append(row)
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(row) + "\n")
        if verbose:
            ev = f" val {row['val']:.2f}" if "val" in row else ""
            hv = f" heldout {row['heldout']:.2f}" if row.get("heldout") is not None else ""
            print(f"[{run_id(cfg)}] t={t} {sched} loss {row['loss']:.4f}{ev}{hv} "
                  f"({train_sec:.0f}s, up {up_params})", flush=True)
        past = deadline is not None and time.time() > deadline
        if t % int(cfg["ckpt_every"]) == 0 or past:
            save_ckpt(t)
        if past and t < int(cfg["rounds"]):
            return {"status": "paused", "round": t}

    ev = [r for r in log if "val" in r]
    best = max(ev, key=lambda r: r["val"])
    hv = [r for r in ev if r.get("heldout") is not None]
    sel_row = max(hv, key=lambda r: (r["heldout"], -r["round"])) if hv else None
    ups = [r["up_params"] for r in log]
    result = dict(
        status="done", run_id=run_id(cfg), cfg=cfg, metric=metric,
        summary=dict(best_val=best["val"], best_val_round=best["round"], final_val=ev[-1]["val"],
                     val_at_best_heldout=None if sel_row is None else sel_row["val"],
                     best_heldout_round=None if sel_row is None else sel_row["round"],
                     up_params_mean=float(np.mean(ups)), up_params_max=int(max(ups)),
                     up_kb_mean=float(np.mean(ups)) * 4 / 1024, rounds=len(log),
                     train_sec=float(sum(r["train_sec"] for r in log)),
                     steps=int(sum(r["steps"] for r in log))),
        sizes=dict(train=len(train), heldout=len(heldout), val=len(val),
                   clients=[len(p) for p in parts][:50]),
        global_params=int(sum(gstate[n].numel() for n in gnames)),
        log=log, env=dict(env_info(device), precision=amp_desc), wall_sec=time.time() - t0_run,
    )
    _write_json(res_path, result)
    for p in (ckpt_path, os.path.join(rdir, "error.txt")):      # error.txt: left over from a failed earlier try
        if os.path.exists(p):
            os.remove(p)
    return result


In [ ]:
%%writefile /content/fedtt_code/fedtt_repro/suites.py
"""Run grids that mirror the experiments of the paper (one suite per table).

Each suite returns a list of run configs (dicts understood by fl.run). The pipeline can filter them
(--tasks/--methods/--lrs/--bss/--seeds) and override the number of rounds (--rounds, which changes the run id).
Anything the paper leaves open is set here and listed in 説明資料.md (§3 "論文に書かれていない設定").
"""
DEB = "microsoft/deberta-base"
RB = "FacebookAI/roberta-base"
RL = "FacebookAI/roberta-large"
GLUE5 = ["mrpc", "sst2", "qnli", "qqp", "mnli"]
GLUE4 = ["sst2", "qnli", "qqp", "mnli"]


def _deb_val(task):
    # Appendix A.1: "For the DeBERTa-Base models on the QQP task, we randomly select 1000 samples from the
    # validation set and report the highest accuracy achieved."
    return dict(val_subsample=1000) if task == "qqp" else {}


def t1(baselines=False):
    """Table 1 (Appendix A.3): DeBERTa-base, cross-silo, IID, 5 clients, 1 local epoch, 100 rounds,
    lr in {5e-3, 1e-3, 5e-4, 1e-4} x batch size in {16, 32}; FedTT and FedTT+ with tensor rank 5."""
    methods = [("fedtt", 5), ("fedttplus", 5)] + ([("lora", 8), ("lora", 4), ("rolora", 4)] if baselines else [])
    out = []
    for task in GLUE5:
        for method, r in methods:
            for lr in (5e-3, 1e-3, 5e-4, 1e-4):
                for bs in (16, 32):
                    out.append(dict(suite="t1", task=task, model=DEB, method=method, rank=r, n_clients=5,
                                    partition="iid", rounds=100, local_epochs=1.0, lr=lr, bs=bs, **_deb_val(task)))
    return out


def t7(baselines=False):
    """Table 7 (Sec. 5.7): DeBERTa-base, 5 clients, one local update (= one local epoch, as in Table 1),
    lr 1e-3, bs 32, best over 20 rounds, tensor rank in {2, 5, 10}. FedTT+ (rank 5) is added in the same
    setting as an extra (not in Table 7)."""
    methods = [("fedtt", 2), ("fedtt", 5), ("fedtt", 10), ("fedttplus", 5)]
    return [dict(suite="t7", task=task, model=DEB, method=m, rank=r, n_clients=5, partition="iid", rounds=20,
                 local_epochs=1.0, lr=1e-3, bs=32, **_deb_val(task))
            for task in GLUE4 for m, r in methods]


def t2(baselines=False):
    """Table 2 (Appendix A.4, FedPETuning setting): RoBERTa-base; MRPC and SST-2 cross-silo with 10 clients,
    QNLI/QQP/MNLI large-scale cross-device with 1000 clients and 10 per round; Dir(1.0) label skew
    (FedPETuning default); 1 local epoch; 100 rounds (cross-silo) / 1000 rounds (LSCD);
    lr in {1e-2, 5e-3, 1e-3, 5e-4, 1e-4, 5e-5} x bs in {16, 32}; FedTT rank 5."""
    methods = [("fedtt", 5)] + ([("lora", 8)] if baselines else [])
    out = []
    for task in GLUE5:
        silo = task in ("mrpc", "sst2")
        base = dict(suite="t2", task=task, model=RB, n_clients=10 if silo else 1000,
                    clients_per_round=None if silo else 10, partition="dir", alpha=1.0,
                    rounds=100 if silo else 1000, local_epochs=1.0, eval_every=1 if silo else 10)
        for method, r in methods:
            for lr in (1e-2, 5e-3, 1e-3, 5e-4, 1e-4, 5e-5):
                for bs in (16, 32):
                    out.append(dict(base, method=method, rank=r, lr=lr, bs=bs))
    return out


def t3(baselines=False):
    """Table 3 (Sec. 5.3, Appendix A.5, RoLoRA setting): RoBERTa-large, 3 / 20 / 50 clients (iid / mild /
    severe heterogeneity, data split uniformly), 20 local updates (steps), lr 1e-3, bs 64 (SST-2) / 32;
    FedTT in the last 6 (SST-2) / 9 (others) layers, FedTT+ in all layers; best over 100 rounds."""
    out = []
    for level, N in (("iid", 3), ("mild", 20), ("severe", 50)):
        for task in GLUE4:
            last = "last6" if task == "sst2" else "last9"
            base = dict(suite="t3", task=task, model=RL, n_clients=N, partition="iid", rounds=100, local_epochs=None,
                        local_steps=20, lr=1e-3, bs=64 if task == "sst2" else 32, tag=level)
            out.append(dict(base, method="fedtt", rank=5, adapter_layers=last))
            out.append(dict(base, method="fedttplus", rank=5, adapter_layers="all"))
            if baselines:
                for m in ("lora", "ffa", "rolora"):
                    out.append(dict(base, method=m, rank=2, adapter_layers=last))
    return out


def t13(baselines=False):
    """Table 13 (Appendix B, FFA-LoRA setting): RoBERTa-large, 3 clients with label-ratio splits
    (iid / mild / severe), 10 local updates, batch size 200, 1000 rounds; FedTT+ in all layers, lr 1e-3."""
    out = []
    for part in ("iid", "mild", "severe"):
        for task in GLUE4:
            base = dict(suite="t13", task=task, model=RL, n_clients=3, partition=part, rounds=1000, local_epochs=None,
                        local_steps=10, bs=200, lr=1e-3, eval_every=10)
            out.append(dict(base, method="fedttplus", rank=5))
            if baselines:
                out.append(dict(base, method="lora", rank=8, lr=1e-3))
                out.append(dict(base, method="ffa", rank=8, lr=1e-3))
    return out


def smoke(baselines=False):
    """Tiny runs on real data/models that exercise every code path (minutes on a GPU)."""
    tiny = dict(suite="smoke", rounds=2, train_subsample=320, val_subsample=96, heldout_subsample=48, ckpt_every=1)
    out = [dict(tiny, task="mrpc", model=DEB, method="fedtt", rank=5, n_clients=5, local_epochs=1.0, bs=32, lr=1e-3),
           dict(tiny, task="mrpc", model=DEB, method="fedttplus", rank=5, n_clients=5, local_epochs=1.0, bs=32, lr=1e-3),
           dict(tiny, task="mnli", model=RB, method="fedtt", rank=5, n_clients=20, clients_per_round=4,
                partition="dir", alpha=1.0, local_epochs=1.0, bs=16, lr=1e-3),
           dict(tiny, task="sst2", model=RL, method="fedttplus", rank=5, n_clients=3, partition="mild",
                local_epochs=None, local_steps=3, bs=16, lr=1e-3),
           dict(tiny, task="qnli", model=RL, method="fedtt", rank=5, adapter_layers="last9", n_clients=3,
                local_epochs=None, local_steps=3, bs=16, lr=1e-3)]
    if baselines:
        out.append(dict(tiny, task="mrpc", model=DEB, method="lora", rank=8, n_clients=5, local_epochs=1.0, bs=32, lr=1e-3))
    return out


SUITES = dict(smoke=smoke, t1=t1, t7=t7, t2=t2, t3=t3, t13=t13)

# Rough training sizes (GLUE train after the FedPETuning split) for cost estimates.
TRAIN_SIZE = dict(mrpc=3301, sst2=66675, qnli=103695, qqp=360210, mnli=388774, rte=2241)
MODEL_COST = {DEB: 1.0, RB: 0.8, RL: 2.8}       # relative time per sample (rough)


def est_samples(cfg):
    """Training samples processed by the whole run (all clients, all rounds)."""
    n = cfg.get("train_subsample") or TRAIN_SIZE[cfg["task"]]
    N = cfg.get("n_clients", 5)
    m = cfg.get("clients_per_round") or N
    if cfg.get("local_steps"):
        per_round = m * cfg["local_steps"] * cfg["bs"]
    else:
        per_round = m * (n / N) * float(cfg.get("local_epochs") or 1.0)
    return per_round * cfg["rounds"]


def est_cost(cfg):
    return est_samples(cfg) * MODEL_COST.get(cfg.get("model"), 1.0)


In [ ]:
%%writefile /content/fedtt_code/fedtt_repro/pipeline.py
"""Runs a suite: skips finished runs, resumes interrupted ones from their checkpoint, optionally in parallel
worker processes, and stops starting new work after --hours.

  python -m fedtt_repro.pipeline --suite t7 --out OUT [--parallel 4] [--hours 20]
         [--tasks sst2,qnli] [--methods fedtt] [--lrs 1e-3] [--bss 32] [--seeds 0,1,2] [--rounds 20]
         [--baselines] [--dry]
"""
import argparse
import json
import os
import subprocess
import sys
import time
import traceback

from . import suites
from .fl import full_cfg, run, run_dir, run_id


def _floats(s):
    return None if not s else [float(x) for x in s.split(",")]


def _ints(s):
    return None if not s else [int(x) for x in s.split(",")]


def _strs(s):
    return None if not s else s.split(",")


def select(args):
    cfgs = suites.SUITES[args.suite](baselines=args.baselines)
    if args.tasks:
        cfgs = [c for c in cfgs if c["task"] in _strs(args.tasks)]
    if args.methods:
        cfgs = [c for c in cfgs if c["method"] in _strs(args.methods)
                or f"{c['method']}-r{c['rank']}" in _strs(args.methods)]
    if args.lrs:
        cfgs = [c for c in cfgs if any(abs(c["lr"] - x) < 1e-12 for x in _floats(args.lrs))]
    if args.bss:
        cfgs = [c for c in cfgs if c["bs"] in _ints(args.bss)]
    if args.levels:
        cfgs = [c for c in cfgs if c.get("tag", "") in _strs(args.levels) or c.get("partition") in _strs(args.levels)]
    seeds = _ints(args.seeds) or [0]
    out = []
    for c in cfgs:
        for s in seeds:
            d = dict(c, seed=s)
            if args.rounds:
                d["rounds"] = int(args.rounds)
            if args.threads:
                d["threads"] = int(args.threads)
            if args.micro_bs:
                d["micro_bs"] = int(args.micro_bs)
            out.append(d)
    return out


def is_done(out_root, cfg):
    return os.path.exists(os.path.join(run_dir(out_root, cfg), "result.json"))


def progress(out_root, cfgs):
    """One line per run in progress: last finished round, its validation score and seconds per round."""
    out = []
    for c in cfgs:
        d = run_dir(out_root, c)
        lp = os.path.join(d, "log.jsonl")
        if os.path.exists(os.path.join(d, "result.json")) or not os.path.exists(lp):
            continue
        try:
            with open(lp, encoding="utf-8") as f:
                rows = [json.loads(x) for x in f if x.strip()]
        except (OSError, ValueError):
            continue
        if not rows:
            continue
        r = rows[-1]
        ev = [x["val"] for x in rows if "val" in x]
        sec = sum(x["train_sec"] + x.get("eval_sec", 0) for x in rows) / len(rows)
        out.append(f"\n    {run_id(c)}: round {r['round']}/{full_cfg(c)['rounds']}, "
                   f"val {ev[-1] if ev else float('nan'):.2f} (best {max(ev) if ev else float('nan'):.2f}), "
                   f"{sec:.0f} s/round")
    return "".join(out)


def worker_main(queue_path, out_root, deadline):
    with open(queue_path, encoding="utf-8") as f:
        cfgs = json.load(f)
    for cfg in cfgs:
        if time.time() > deadline:
            print("deadline reached; stopping", flush=True)
            return
        if is_done(out_root, cfg):
            continue
        try:
            r = run(cfg, out_root, deadline=deadline)
            if r.get("status") == "paused":
                print(f"paused {run_id(cfg)} at round {r['round']}", flush=True)
                return
            s = r["summary"]
            print(f"DONE {run_id(cfg)}: best val {s['best_val']:.2f} (round {s['best_val_round']}), "
                  f"val@best-heldout {s['val_at_best_heldout']}, {s['train_sec'] / 60:.1f} min", flush=True)
        except Exception:
            err = traceback.format_exc()
            print(f"ERROR {run_id(cfg)}\n{err}", flush=True)
            d = run_dir(out_root, cfg)
            os.makedirs(d, exist_ok=True)
            with open(os.path.join(d, "error.txt"), "a", encoding="utf-8") as f:
                f.write(time.strftime("%Y-%m-%d %H:%M:%S ") + err + "\n")


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--suite", default="smoke", choices=sorted(suites.SUITES))
    ap.add_argument("--out", required=True)
    ap.add_argument("--parallel", type=int, default=1)
    ap.add_argument("--hours", type=float, default=1e6)
    ap.add_argument("--tasks"), ap.add_argument("--methods"), ap.add_argument("--lrs"), ap.add_argument("--bss")
    ap.add_argument("--levels", help="t3/t13: iid,mild,severe")
    ap.add_argument("--seeds"), ap.add_argument("--rounds", type=int), ap.add_argument("--threads", type=int)
    ap.add_argument("--micro_bs", type=int, help="split each minibatch into chunks (gradient accumulation; GPU memory)")
    ap.add_argument("--baselines", action="store_true", help="also run the LoRA-family references")
    ap.add_argument("--dry", action="store_true", help="list the runs and the estimated work, run nothing")
    ap.add_argument("--worker", help=argparse.SUPPRESS)
    ap.add_argument("--deadline", type=float, help=argparse.SUPPRESS)
    args = ap.parse_args(argv)
    if args.worker:
        return worker_main(args.worker, args.out, args.deadline)

    cfgs = select(args)
    todo = [c for c in cfgs if not is_done(args.out, c)]
    todo.sort(key=lambda c: -suites.est_cost(c))
    tot = sum(suites.est_samples(c) for c in todo)
    print(f"suite {args.suite}: {len(cfgs)} runs, {len(cfgs) - len(todo)} done, {len(todo)} to go "
          f"({tot / 1e6:.1f} M training samples)", flush=True)
    if args.dry:
        for c in todo:
            print(f"  {run_id(c)}  ~{suites.est_samples(c) / 1e6:.2f} M samples")
        return
    if not todo:
        return
    deadline = time.time() + 3600 * args.hours
    os.makedirs(os.path.join(args.out, "queue"), exist_ok=True)
    os.makedirs(os.path.join(args.out, "logs"), exist_ok=True)
    if args.parallel <= 1:
        qp = os.path.join(args.out, "queue", f"{args.suite}_w0.json")
        with open(qp, "w", encoding="utf-8") as f:
            json.dump([full_cfg(c) for c in todo], f)
        return worker_main(qp, args.out, deadline)
    # greedy balance by estimated cost; each worker runs its queue in order
    shards, load = [[] for _ in range(args.parallel)], [0.0] * args.parallel
    for c in todo:
        j = load.index(min(load))
        shards[j].append(full_cfg(c))
        load[j] += suites.est_cost(c)
    procs, logs = [], []
    for j, sh in enumerate(shards):
        if not sh:
            continue
        qp = os.path.join(args.out, "queue", f"{args.suite}_w{j}.json")
        with open(qp, "w", encoding="utf-8") as f:
            json.dump(sh, f)
        logs.append(os.path.join(args.out, "logs", f"{args.suite}_w{j}.log"))
        lf = open(logs[-1], "w", encoding="utf-8")
        cmd = [sys.executable, "-m", "fedtt_repro.pipeline", "--suite", args.suite, "--out", args.out,
               "--worker", qp, "--deadline", str(deadline)]
        procs.append((subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT), lf))
        print(f"worker {j}: {len(sh)} runs -> logs/{args.suite}_w{j}.log", flush=True)
    while any(p.poll() is None for p, _ in procs):
        time.sleep(60)
        done = sum(is_done(args.out, c) for c in cfgs)
        print(time.strftime("%H:%M:%S"), f"{done}/{len(cfgs)} runs done" + progress(args.out, todo), flush=True)
    for _, lf in procs:
        lf.close()
    done = sum(is_done(args.out, c) for c in cfgs)
    print(f"finished: {done}/{len(cfgs)} runs done", flush=True)
    show_errors(logs)


def show_errors(log_paths, n_lines=60):
    """Print the end of every worker log that contains an error (worker output goes to files, not here)."""
    for lp in log_paths:
        try:
            with open(lp, encoding="utf-8", errors="replace") as f:
                text = f.read()
        except OSError:
            continue
        if "ERROR" in text or "Traceback" in text:
            start = max(text.rfind("ERROR"), text.rfind("Traceback"))
            tail = text[max(0, start - 200):].splitlines()[-n_lines:]
            print(f"\n----- errors in {lp} -----\n" + "\n".join(tail), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/fedtt_code/fedtt_repro/count.py
"""Exact trainable-parameter and uplink counts of our implementation vs. the paper (Tables 1, 2, 7, 12, 13-15).

No training and no weights: the models are built from their Hugging Face configs (vocabulary shrunk, which
does not change any counted parameter). A count "matches" when it rounds to the paper's printed value
(0.06M = [0.055M, 0.065M), 39K = [38.5K, 39.5K)). Uplink KB = params * 4 B / 1024 (fp32), as in Table 14.

  python -m fedtt_repro.count [--out OUT/report/param_check.md]
"""
import argparse
import os
import re

from fedtt import model as M
from .suites import DEB, RB, RL

PAPER = [
    # (label, model, n_cls, method, rank, layers, paper value, source)
    ("DeBERTa-base FedTT r5 (binary)", DEB, 2, "fedtt", 5, "all", "0.06M", "Table 1 / 7"),
    ("DeBERTa-base FedTT r5 (MNLI)", DEB, 3, "fedtt", 5, "all", "0.06M", "Table 1"),
    ("DeBERTa-base FedTT+ r5 (binary)", DEB, 2, "fedttplus", 5, "all", "0.02M", "Table 1"),
    ("DeBERTa-base FedTT r2", DEB, 2, "fedtt", 2, "all", "0.03M", "Table 7"),
    ("DeBERTa-base FedTT r10", DEB, 2, "fedtt", 10, "all", "0.17M", "Table 7"),
    ("DeBERTa-base LoRA r8 (head not counted)", DEB, 2, "lora", 8, "all", "0.30M", "Table 1"),
    ("DeBERTa-base LoRA r4 (head not counted)", DEB, 2, "lora", 4, "all", "0.15M", "Table 1"),
    ("RoBERTa-base FedTT r5", RB, 2, "fedtt", 5, "all", "0.06M", "Table 2"),
    ("RoBERTa-large FedTT r5, last 6 (SST-2)", RL, 2, "fedtt", 5, "last6", "39K", "Table 12"),
    ("RoBERTa-large FedTT r5, last 9 (QNLI)", RL, 2, "fedtt", 5, "last9", "55K", "Table 12"),
    ("RoBERTa-large FedTT r5, last 9 (MNLI)", RL, 3, "fedtt", 5, "last9", "55K", "Table 12"),
    ("RoBERTa-large FedTT+ r5, all (SST-2)", RL, 2, "fedttplus", 5, "all", "28K", "Table 12"),
    ("RoBERTa-large FedTT+ r5, all (MNLI)", RL, 3, "fedttplus", 5, "all", "28K", "Table 12"),
    ("RoBERTa-large FedTT+ r5, all (binary)", RL, 2, "fedttplus", 5, "all", "0.03M", "Table 13"),
    ("RoBERTa-large LoRA r2, last 9 (head not counted)", RL, 2, "lora", 2, "last9", "74K", "Table 12"),
]


def parse(v):
    num, unit = float(v[:-1]), v[-1]
    return num * {"M": 1e6, "K": 1e3}[unit], unit, len(v.split(".")[1][:-1]) if "." in v else 0


def matches(n, v):
    x, unit, dec = parse(v)
    scale = {"M": 1e6, "K": 1e3}[unit]
    return round(n / scale, dec) == round(x / scale, dec)


def fmt(n, v):
    _, unit, dec = parse(v)
    scale = {"M": 1e6, "K": 1e3}[unit]
    return f"{n / scale:.{max(dec, 1) + 1}f}{unit}"


def config(name, n_cls):
    from transformers import AutoConfig
    c = AutoConfig.from_pretrained(name, num_labels=n_cls)
    c.vocab_size = 8           # embeddings are frozen and never counted; keeps the build fast
    return c


def round_counts(fm, method, head=True):
    """Counts for rounds 1..6 (FedTT+ cycles through r; RoLoRA alternates)."""
    out = []
    for t in range(1, 7):
        fm.set_round(t)
        n = 0
        for name, p in fm.trainable():
            if not head and (name in fm.head_names or ".modules_to_save." in name):
                continue
            n += p.numel()
        out.append(n)
    return out


def build(name, n_cls, method, rank, layers):
    cfg = dict(model=name, method=method, rank=rank, adapter_layers=layers)
    return M.build_model(cfg, n_cls, hf_config=config(name, n_cls), pretrained=False)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--out")
    args = ap.parse_args(argv)
    lines = ["# Trainable parameters / uplink per round: our implementation vs. the paper", "",
             "| setting | source | paper | ours (per round) | ours KB/round | match | breakdown (round 1) |",
             "|---|---|---|---|---|---|---|"]
    for label, name, n_cls, method, rank, layers, paper, src in PAPER:
        try:
            fm = build(name, n_cls, method, rank, layers)
        except Exception as e:   # e.g. peft refusing Colab's old torchao for the LoRA rows; keep the other rows
            err = f"{type(e).__name__}: {e}".replace("|", "/").replace("\n", " ")[:120]
            lines.append(f"| {label} | {src} | {paper} | not built ({err}) | | | |")
            print(lines[-1], flush=True)
            continue
        head = "head not counted" not in label
        cnt = round_counts(fm, method, head)
        lo, hi = min(cnt), max(cnt)
        mean = sum(cnt[:6]) / 6
        shown = f"{lo:,}" if lo == hi else f"{lo:,}-{hi:,} (mean {mean:,.0f})"
        ok = matches(mean, paper) or matches(hi, paper) or matches(lo, paper)
        okstr = "yes" if matches(mean, paper) else ("partly (min/max)" if ok else "**no**")
        bd = M.param_breakdown(fm, 1)
        bds = ", ".join(f"{k} {v:,}" for k, v in bd.items() if v)
        lines.append(f"| {label} | {src} | {paper} | {shown} = {fmt(mean, paper)} | {mean * 4 / 1024:.0f} | "
                     f"{okstr} | {bds} |")
        print(lines[-1], flush=True)
    # what FedTT+ would count if the adapter biases were trained too (they are frozen by default; see README)
    lines += ["", "FedTT+ variants (DeBERTa-base, binary):", ""]
    for tb in (False, True):
        fm = M.build_model(dict(model=DEB, method="fedttplus", rank=5, plus_train_bias=tb), 2,
                           hf_config=config(DEB, 2), pretrained=False)
        cnt = round_counts(fm, "fedttplus")
        lines.append(f"- plus_train_bias={tb}: rounds 1-6 {cnt} -> mean {sum(cnt) / 6:,.0f}")
    # Table 14/15 KB lines
    lines += ["", "Paper uplink KB (Table 14/15) are 4 B x the rounded counts / 1024: 0.06M -> 234, 0.02M -> 78, "
              "51K -> 199, 28K -> 109.", ""]
    text = "\n".join(lines) + "\n"
    print("\n".join(lines[-6:]))
    if args.out:
        os.makedirs(os.path.dirname(args.out), exist_ok=True)
        with open(args.out, "w", encoding="utf-8") as f:
            f.write(text)
    return text


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/fedtt_code/fedtt_repro/report.py
"""Collects finished runs and compares them with the paper.

For every (suite, setting, method, task):
  best    = best GLUE-validation score over rounds, maximised over the learning-rate / batch-size grid
            (the paper's protocol as written: "we record the best validation results"; optimistic);
  sel     = the GLUE-validation score at the round and grid point with the best held-out (FedPETuning dev)
            score (selection without looking at the evaluation set);
  paper   = the paper's number, delta = best - paper;
  r95     = first round whose validation score reaches 95 % of the paper's score (Tables 14/15 definition).
With several seeds the mean (and std) over seeds is shown.

  python -m fedtt_repro.report --out OUT
"""
import argparse
import glob
import json
import os

import numpy as np

from . import paper

TASKS = paper.TASK_ORDER


def load_runs(out_root):
    rows = []
    for p in glob.glob(os.path.join(out_root, "runs", "*", "*", "*", "result.json")):
        with open(p, encoding="utf-8") as f:
            r = json.load(f)
        c = r["cfg"]
        rows.append(dict(suite=c["suite"], task=c["task"], method=c["method"], rank=c["rank"],
                         key=f"{c['method']}-r{c['rank']}", level=_level(c), lr=c["lr"], bs=c["bs"], seed=c["seed"],
                         rounds=c["rounds"], best=r["summary"]["best_val"], best_round=r["summary"]["best_val_round"],
                         sel=r["summary"]["val_at_best_heldout"],
                         heldout_best=max([x["heldout"] for x in r["log"] if x.get("heldout") is not None],
                                          default=None),
                         kb=r["summary"]["up_kb_mean"], up=r["summary"]["up_params_mean"],
                         curve=[(x["round"], x["val"]) for x in r["log"] if "val" in x],
                         hours=r["summary"]["train_sec"] / 3600, gpu=r["env"].get("gpu", r["env"].get("cpu", "")),
                         run_id=r["run_id"]))
    return rows


def _level(c):
    if c["suite"] == "t3":
        return c.get("tag") or {3: "iid", 20: "mild", 50: "severe"}.get(c["n_clients"], "")
    if c["suite"] == "t13":
        return c["partition"]
    return ""


def paper_value(suite, level, key, task):
    tab = {"t1": paper.T1, "t2": paper.T2, "t7": paper.T7}.get(suite)
    if suite == "t3":
        tab = paper.T3.get(level, {})
    if suite == "t13":
        tab = paper.T13.get(level, {})
    if tab is None:
        return None
    return tab.get(key, {}).get(task)


def r95(curve, target):
    if target is None:
        return None
    for t, v in curve:
        if v >= 0.95 * target:
            return t
    return None


def summarize(rows):
    """One entry per (suite, level, key, task): grid-maximised and held-out-selected scores over seeds."""
    out = {}
    groups = {}
    for r in rows:
        groups.setdefault((r["suite"], r["level"], r["key"], r["task"], r["rounds"]), []).append(r)
    for (suite, level, key, task, rounds), rs in groups.items():
        seeds = sorted({r["seed"] for r in rs})
        best, sel, rr, kb, hrs = [], [], [], [], []
        target = paper_value(suite, level, key, task)
        for s in seeds:
            g = [r for r in rs if r["seed"] == s]
            b = max(g, key=lambda r: r["best"])
            best.append(b["best"])
            hs = [r for r in g if r["heldout_best"] is not None]
            if hs:
                sel.append(max(hs, key=lambda r: r["heldout_best"])["sel"])
            rr.append(r95(b["curve"], target))
            kb.append(b["kb"])
            hrs.append(sum(r["hours"] for r in g))
        out[(suite, level, key, task, rounds)] = dict(
            best=float(np.mean(best)), best_std=float(np.std(best)) if len(best) > 1 else None,
            sel=float(np.mean(sel)) if sel else None, n_seeds=len(seeds), n_grid=len(rs) // len(seeds),
            paper=target, r95=rr[0] if len(rr) == 1 else rr, kb=float(np.mean(kb)), hours=float(np.sum(hrs)),
            best_cfg=f"lr {max(rs, key=lambda r: r['best'])['lr']:g}, bs {max(rs, key=lambda r: r['best'])['bs']}")
    return out


def _f(x, d=2):
    return "–" if x is None else f"{x:.{d}f}"


def tables(summary):
    lines = []
    suites = sorted({k[0] for k in summary}, key=lambda s: ["smoke", "t1", "t7", "t2", "t3", "t13"].index(s)
                    if s in ["smoke", "t1", "t7", "t2", "t3", "t13"] else 99)
    titles = dict(t1="Table 1 (DeBERTa-base, 5 clients, IID)", t7="Table 7 (DeBERTa-base, rank analysis, 20 rounds)",
                  t2="Table 2 (RoBERTa-base, FedPETuning setting)", t3="Table 3 (RoBERTa-large, 3/20/50 clients)",
                  t13="Table 13 (RoBERTa-large, 3 clients, label skew)", smoke="smoke (pipeline check only)")
    for suite in suites:
        keys = [k for k in summary if k[0] == suite]
        lines += [f"## {titles.get(suite, suite)}", ""]
        for level in sorted({k[1] for k in keys}, key=lambda x: ["", "iid", "mild", "severe"].index(x)
                            if x in ["", "iid", "mild", "severe"] else 9):
            for rounds in sorted({k[4] for k in keys if k[1] == level}):
                ks = [k for k in keys if k[1] == level and k[4] == rounds]
                tasks = [t for t in TASKS if any(k[3] == t for k in ks)]
                if level:
                    lines += [f"### {level}", ""]
                lines += [f"rounds = {rounds}. Each cell: **ours best** / paper (Δ) ; held-out-selected ; r95.", ""]
                lines += ["| method | " + " | ".join(tasks) + " | avg ours | avg paper |",
                          "|---|" + "---|" * (len(tasks) + 2)]
                for key in sorted({k[2] for k in ks}):
                    cells, ours, pap = [], [], []
                    for t in tasks:
                        s = summary.get((suite, level, key, t, rounds))
                        if s is None:
                            cells.append("–")
                            continue
                        d = None if s["paper"] is None else s["best"] - s["paper"]
                        std = f"±{s['best_std']:.2f}" if s["best_std"] else ""
                        cells.append(f"**{s['best']:.2f}**{std} / {_f(s['paper'])} ({'' if d is None else f'{d:+.2f}'})"
                                     f" ; {_f(s['sel'])} ; {s['r95'] if s['r95'] is not None else '–'}")
                        ours.append(s["best"])
                        if s["paper"] is not None:
                            pap.append(s["paper"])
                    full = len(ours) == len(tasks)
                    lines.append(f"| {key} | " + " | ".join(cells) +
                                 f" | {_f(np.mean(ours)) if full and ours else '–'} | "
                                 f"{_f(np.mean(pap)) if len(pap) == len(tasks) else '–'} |")
                lines.append("")
                lines += ["| method | " + " | ".join(tasks) + " |", "|---|" + "---|" * len(tasks)]
                for key in sorted({k[2] for k in ks}):
                    cells = []
                    for t in tasks:
                        s = summary.get((suite, level, key, t, rounds))
                        cells.append("–" if s is None else f"{s['best_cfg']}; {s['n_grid']} cfg × {s['n_seeds']} seed; "
                                     f"{s['kb']:.0f} KB/round; {s['hours']:.1f} h")
                    lines.append(f"| {key} | " + " | ".join(cells) + " |")
                lines.append("")
    return lines


def plot_curves(rows, path):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except Exception:
        return None
    groups = {}
    for r in rows:
        if r["suite"] in ("t3", "t7", "t1") and r["seed"] == 0:
            groups.setdefault((r["suite"], r["level"], r["task"]), []).append(r)
    if not groups:
        return None
    keys = sorted(groups)
    n = len(keys)
    cols = 4
    rows_n = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(3.2 * cols, 2.5 * rows_n), squeeze=False)
    for ax, k in zip(axes.flat, keys):
        best = {}
        for r in groups[k]:
            if r["key"] not in best or r["best"] > best[r["key"]]["best"]:
                best[r["key"]] = r
        for key, r in sorted(best.items()):
            t, v = zip(*r["curve"])
            ax.plot(t, v, label=key, lw=1.2)
        ax.set_title(f"{k[0]} {k[1]} {k[2]}".replace("  ", " "), fontsize=8)
        ax.tick_params(labelsize=7)
        ax.legend(fontsize=6)
    for ax in list(axes.flat)[n:]:
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)
    return path


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", required=True)
    args = ap.parse_args(argv)
    rows = load_runs(args.out)
    os.makedirs(os.path.join(args.out, "report"), exist_ok=True)
    if not rows:
        print("no finished runs yet")
        return
    summ = summarize(rows)
    lines = ["# FedTT / FedTT+ reproduction: our runs vs. the paper", "",
             f"{len(rows)} finished runs. 'best' = best GLUE-validation score over rounds, maximised over the "
             "lr/bs grid (paper protocol); 'held-out-selected' = validation score at the round/grid point with the "
             "best FedPETuning held-out score; r95 = first round reaching 95 % of the paper's score.", ""]
    lines += tables(summ)
    fig = plot_curves(rows, os.path.join(args.out, "report", "curves.png"))
    if fig:
        lines += [f"Curves (validation score per round, best grid point, seed 0): `{os.path.basename(fig)}`", ""]
    text = "\n".join(lines) + "\n"
    with open(os.path.join(args.out, "report", "report.md"), "w", encoding="utf-8") as f:
        f.write(text)
    flat = [dict(suite=k[0], level=k[1], method=k[2], task=k[3], rounds=k[4], **{kk: vv for kk, vv in v.items()
                                                                                    if kk != "r95"}, r95=str(v["r95"]))
            for k, v in summ.items()]
    import csv
    with open(os.path.join(args.out, "report", "summary.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(flat[0]))
        w.writeheader()
        w.writerows(flat)
    print(text)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/fedtt_code/fedtt_repro/paper.py
"""Numbers reported by Ghiasvand et al., "Communication-Efficient and Tensorized Federated Fine-Tuning of Large
Language Models", Findings of ACL 2025, pp. 24192-24207 (transcribed from the published PDF, 2026-11-24).

Scores: MRPC F1, others accuracy (%). Column order in the paper differs between tables; here keys are tasks.
Rows marked 'ours: no' were copied by the authors from other papers (FedPETuning, RoLoRA, FFA-LoRA).
"""

T1 = {  # Table 1: DeBERTa-base, cross-silo, IID, 5 clients, 1 local epoch, best over 100 rounds
    "lora-r8":    dict(params="0.30M", mrpc=91.87, sst2=94.95, qnli=92.68, qqp=89.2, mnli=87.31, avg=91.10),
    "p-tuning":   dict(params="0.30M", mrpc=82.01, sst2=90.48, qnli=82.12, qqp=84.0, mnli=80.74, avg=83.87),
    "lora-r4":    dict(params="0.15M", mrpc=91.72, sst2=94.95, qnli=92.66, qqp=86.7, mnli=86.91, avg=90.58),
    "bitfit":     dict(params="0.10M", mrpc=91.33, sst2=94.72, qnli=91.89, qqp=88.4, mnli=86.02, avg=90.47),
    "rolora-r4":  dict(params="0.08M", mrpc=91.17, sst2=94.61, qnli=92.40, qqp=87.9, mnli=86.27, avg=90.47),
    "prompt":     dict(params="0.01M", mrpc=82.96, sst2=92.32, qnli=82.13, qqp=80.5, mnli=74.46, avg=82.47),
    "fedtt-r5":     dict(params="0.06M", mrpc=92.68, sst2=94.61, qnli=92.02, qqp=88.4, mnli=85.99, avg=90.74),
    "fedttplus-r5": dict(params="0.02M", mrpc=92.60, sst2=93.58, qnli=90.54, qqp=87.9, mnli=85.33, avg=89.99),
}

T2 = {  # Table 2: RoBERTa-base; MRPC/SST-2 cross-silo 10 clients, others LSCD 1000 clients (10/round)
    "prefix":   dict(params="3.50M", mrpc=88.1, sst2=93.7, qnli=84.6, qqp=81.8, mnli=80.4, avg=85.7),
    "p-tuning": dict(params="0.88M", mrpc=86.8, sst2=92.1, qnli=85.6, qqp=81.5, mnli=79.8, avg=85.2),
    "adapter":  dict(params="0.70M", mrpc=88.5, sst2=94.0, qnli=85.9, qqp=87.0, mnli=84.9, avg=88.1),
    "ia3":      dict(params="0.65M", mrpc=88.0, sst2=93.0, qnli=89.4, qqp=85.4, mnli=82.7, avg=87.7),
    "lora":     dict(params="0.30M", mrpc=89.8, sst2=94.4, qnli=86.0, qqp=86.5, mnli=84.7, avg=88.3),
    "bitfit":   dict(params="0.10M", mrpc=88.6, sst2=92.8, qnli=80.5, qqp=84.0, mnli=80.7, avg=85.3),
    "fedtt-r5": dict(params="0.06M", mrpc=88.9, sst2=93.8, qnli=88.9, qqp=86.2, mnli=84.2, avg=88.4),
}

# Table 3: RoBERTa-large, 3 / 20 / 50 clients (iid / mild / severe), 20 local updates. FedTT adapters in the
# last 6 (SST-2) / 9 (others) layers, FedTT+ in all layers. NOTE: the LoRA/FFA-LoRA/RoLoRA rows equal RoLoRA's
# Table 2 with the QQP and MNLI columns swapped (RoLoRA orders SST-2, QNLI, MNLI, QQP); kept as printed.
T3 = {
    "iid": {
        "lora-r2":      dict(params="68K", sst2=95.64, qnli=92.04, qqp=85.85, mnli=86.16, avg=89.92),
        "ffa-r2":       dict(params="34K", sst2=94.91, qnli=90.11, qqp=84.06, mnli=85.48, avg=88.64),
        "rolora-r2":    dict(params="34K", sst2=95.60, qnli=91.62, qqp=85.66, mnli=86.16, avg=89.76),
        "fedtt-r5":     dict(params="51K", sst2=94.38, qnli=93.01, qqp=88.30, mnli=87.20, avg=90.72),
        "fedttplus-r5": dict(params="28K", sst2=95.64, qnli=94.05, qqp=88.99, mnli=88.27, avg=91.74),
    },
    "mild": {
        "lora-r2":      dict(params="68K", sst2=94.27, qnli=86.91, qqp=81.22, mnli=82.07, avg=86.12),
        "ffa-r2":       dict(params="34K", sst2=93.92, qnli=89.58, qqp=80.51, mnli=82.62, avg=86.66),
        "rolora-r2":    dict(params="34K", sst2=94.84, qnli=90.77, qqp=85.13, mnli=85.10, avg=88.96),
        "fedtt-r5":     dict(params="51K", sst2=94.15, qnli=91.38, qqp=86.25, mnli=86.53, avg=89.58),
        "fedttplus-r5": dict(params="28K", sst2=95.64, qnli=92.60, qqp=87.76, mnli=88.11, avg=91.03),
    },
    "severe": {
        "lora-r2":      dict(params="68K", sst2=93.23, qnli=82.57, qqp=58.96, mnli=76.96, avg=77.93),
        "ffa-r2":       dict(params="34K", sst2=92.32, qnli=85.15, qqp=62.79, mnli=77.78, avg=79.51),
        "rolora-r2":    dict(params="34K", sst2=94.61, qnli=89.83, qqp=85.15, mnli=85.55, avg=88.78),
        "fedtt-r5":     dict(params="51K", sst2=94.38, qnli=90.55, qqp=85.47, mnli=85.27, avg=88.92),
        "fedttplus-r5": dict(params="28K", sst2=94.50, qnli=90.17, qqp=86.65, mnli=86.28, avg=89.40),
    },
}
T3_CLIENTS = {"iid": 3, "mild": 20, "severe": 50}

T7 = {  # Table 7: DeBERTa-base, 5 clients, 1 local update (epoch), lr 1e-3, bs 32, best over 20 rounds
    "fedtt-r2":  dict(params="0.03M", sst2=93.58, qnli=90.76, qqp=86.79, mnli=85.29, avg=89.10),
    "fedtt-r5":  dict(params="0.06M", sst2=93.00, qnli=91.87, qqp=87.65, mnli=85.91, avg=89.61),
    "fedtt-r10": dict(params="0.17M", sst2=93.69, qnli=92.48, qqp=88.22, mnli=87.01, avg=90.35),
}

T13 = {  # Table 13 (Appendix B): RoBERTa-large, 3 clients, label-ratio splits, 10 local updates
    "iid": {"lora-r8": dict(params="1.57M", sst2=94.42, qnli=91.38, qqp=84.47, mnli=86.90, avg=89.29),
            "ffa-r8": dict(params="0.79M", sst2=95.14, qnli=92.64, qqp=86.31, mnli=87.13, avg=90.30),
            "fedttplus-r5": dict(params="0.03M", sst2=95.41, qnli=94.05, qqp=88.15, mnli=88.65, avg=91.56)},
    "mild": {"lora-r8": dict(params="1.57M", sst2=93.55, qnli=91.36, qqp=84.41, mnli=87.01, avg=89.08),
             "ffa-r8": dict(params="0.79M", sst2=94.10, qnli=91.62, qqp=85.33, mnli=87.04, avg=89.52),
             "fedttplus-r5": dict(params="0.03M", sst2=95.53, qnli=93.54, qqp=87.91, mnli=88.45, avg=91.36)},
    "severe": {"lora-r8": dict(params="1.57M", sst2=94.32, qnli=88.95, qqp=83.51, mnli=82.03, avg=87.20),
               "ffa-r8": dict(params="0.79M", sst2=94.32, qnli=90.35, qqp=84.35, mnli=85.05, avg=88.52),
               "fedttplus-r5": dict(params="0.03M", sst2=95.64, qnli=91.67, qqp=86.66, mnli=87.73, avg=90.42)},
}

# Table 12: trainable parameters for Table 3 (RoBERTa-large)
T12 = {"lora-r2": dict(sst2="49K", qnli="74K", qqp="74K", mnli="74K"),
       "ffa-r2": dict(sst2="25K", qnli="37K", qqp="37K", mnli="37K"),
       "rolora-r2": dict(sst2="25K", qnli="37K", qqp="37K", mnli="37K"),
       "fedtt-r5": dict(sst2="39K", qnli="55K", qqp="55K", mnli="55K"),
       "fedttplus-r5": dict(sst2="28K", qnli="28K", qqp="28K", mnli="28K")}

# Table 14 (Table 1 setting) and Table 15 (Table 3 severe): uplink KB per round and rounds to reach 95 % of the
# Table 1 / Table 3 score. KB = params * 4 B / 1024 computed from the rounded counts (0.06M -> 234 KB).
T14 = {"lora-r4": dict(kb=586, rounds=dict(sst2=2, qnli=2, qqp=2, mnli=2)),
       "rolora-r4": dict(kb=312, rounds=dict(sst2=2, qnli=2, qqp=1, mnli=2)),
       "fedtt-r5": dict(kb=234, rounds=dict(sst2=2, qnli=2, qqp=2, mnli=2)),
       "fedttplus-r5": dict(kb=78, rounds=dict(sst2=5, qnli=6, qqp=3, mnli=3))}
T15 = {"rolora-r2": dict(kb=133, rounds=dict(sst2=3, qnli=10, qqp=11, mnli=11)),
       "fedtt-r5": dict(kb=199, rounds=dict(sst2=3, qnli=3, qqp=3, mnli=3)),
       "fedttplus-r5": dict(kb=109, rounds=dict(sst2=3, qnli=3, qqp=3, mnli=3))}

TASK_ORDER = ["mrpc", "sst2", "qnli", "qqp", "mnli"]


## 準備

In [ ]:
import json, glob
os.chdir(CODE)
sys.path.insert(0, CODE)

def sh(cmd):
    """コマンドを実行し、出力をこのセルに流す（ダウンロードの進捗表示などは省く）。"""
    noise = ('it/s]', 's/it]', 'Loading weights', 'LOAD REPORT', 'UNEXPECTED', 'MISSING', 'Notes:', 'can be ignored',
             'newly initialized', '---+---', 'Key ', 'HF_TOKEN', 'Redirects are currently')
    env = dict(os.environ, PYTHONIOENCODING='utf-8', PYTHONUNBUFFERED='1')
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         encoding='utf-8', errors='replace', env=env)
    for line in p.stdout:
        if not any(s in line for s in noise):
            print(line, end='', flush=True)
    return p.wait()

def step(name, args, max_par=6):
    left = (T_END - time.time()) / 3600
    if left < 0.1:
        print(f'時間切れのため「{name}」は始めません（もう一度「すべてのセルを実行」で続きから）'); return False
    print(f'\n===== {name}（残り {left:.1f} 時間） =====', flush=True)
    sh(f'{sys.executable} -m fedtt_repro.pipeline {args} --out "{OUT}" --parallel {min(PARALLEL, max_par)} --hours {left:.2f}')
    return True

r = sh(f'{sys.executable} -c "import fedtt_repro.fl, fedtt_repro.pipeline; print(\'code ok\')"')
assert r == 0, 'コードの読み込みに失敗しました（上のエラーを見てください）'

## 実行（パラメータ数の照合 → 各ステップ）

In [ ]:
# (名前, pipeline の引数, 同時に流す本数の上限)。不要な行を消せば飛ばせます。
STEPS = [
    ('1 スモーク',           '--suite smoke', 6),
    ('2 表1 MRPC',           '--suite t1 --tasks mrpc', 6),
    ('3 表7 SST-2/QNLI',     '--suite t7 --tasks sst2,qnli', 6),
    ('4 表3 iid',            '--suite t3 --levels iid', 6),
    ('5 表7 QQP/MNLI',       '--suite t7 --tasks qqp,mnli', 6),
    ('6 表3 mild/severe',    '--suite t3 --levels mild,severe', 6),
    ('7 表2',                '--suite t2 --lrs 5e-3,1e-3 --bss 32', 6),
    ('8 表13',               '--suite t13 --micro_bs 50', 3),      # バッチ 200 を 50 ずつに分けて勾配を足す（結果は同じ）
]
sh(f'{sys.executable} -m fedtt_repro.count --out "{OUT}/report/param_check.md"')
for name, args, max_par in STEPS:
    if not step(name, args, max_par):
        break

## 集計（論文との比較）

In [ ]:
sh(f'{sys.executable} -m fedtt_repro.report --out "{OUT}"')
from IPython.display import display, Markdown, Image
if os.path.exists(f'{OUT}/report/curves.png'):
    display(Image(f'{OUT}/report/curves.png'))
print('論文との比較表:', f'{OUT}/report/report.md', '| パラメータ数の照合:', f'{OUT}/report/param_check.md')

## うまくいかないとき
ランが終わらない・エラーが出るときは、このセルを実行して、出てきた表示をそのまま共有してください。
各ワーカーのログの最後と、失敗したランのエラー（`error.txt`）、各ランの最新のラウンドを表示します。

In [ ]:
import glob, json, os
for f in sorted(glob.glob(f'{OUT}/logs/*.log')):
    print('=====', f)
    print(open(f, encoding='utf-8', errors='replace').read()[-2500:])
errs = glob.glob(f'{OUT}/runs/**/error.txt', recursive=True)
print(f'\n失敗したランの記録: {len(errs)} 件')
for f in errs[:3]:
    print('===== ERROR FILE', f)
    print(open(f, encoding='utf-8', errors='replace').read()[-2500:])
for f in sorted(glob.glob(f'{OUT}/runs/**/log.jsonl', recursive=True)):
    rows = [json.loads(x) for x in open(f, encoding='utf-8') if x.strip()]
    if rows:
        r = rows[-1]
        print(os.path.basename(os.path.dirname(f)), '| round', r['round'], '| val', round(r.get('val', float('nan')), 2))